# Urban Flood Impact Simulation: Road Network Routing Under Dry and Flooded Conditions

**Purpose:** This notebook simulates vehicle travel times across an urban road network under two conditions:
1. **Dry (baseline):** Normal traffic routing with no flood disruption.
2. **Wet (flood scenarios):** Routing on networks disrupted by flooding at ten return period thresholds (5–1000 years).

The simulation uses a Monte Carlo convergence approach to sample Origin–Destination (OD) pairs weighted by population density, computing shortest-path travel times via OSMnx and NetworkX. Results feed downstream resilience and accessibility analyses.

---

**Inputs:**
- Pre-built road network graphs (pickled NetworkX/OSMnx graphs) — dry and flood-disrupted variants
- OD probability point files (CSV) encoding population-weighted sampling grids

**Outputs:**
- Per-network dry routing CSVs (`dry_OD_routing/`)
- Per-network combined dry + wet routing CSVs (`2576_dry_wet_OD_routing/`)
- Summary DataFrame of mean dry travel times per network

## Section 1. Import Libraries

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────────
import os
import sys
import shutil
import random
import pickle

# ── Progress bars ───────────────────────────────────────────────────────────────
from tqdm import tqdm

# ── Geospatial ──────────────────────────────────────────────────────────────────
import osmnx as ox
import networkx as nx
import geopandas as gpd
import shapely

# ── Data science ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotting ────────────────────────────────────────────────────────────────────
from matplotlib import pyplot as plt

# ── Display configuration ───────────────────────────────────────────────────────
import warnings; warnings.simplefilter('ignore')
pd.options.display.max_columns = None
pd.options.display.max_rows = 30
np.set_printoptions(threshold=sys.maxsize)

## Section 2. Dry Condition Routing

This section computes baseline (dry, no-flood) travel times for each road network sub-graph. 
For each network:
1. The graph is loaded and enriched with posted speed limits and edge-level travel times.
2. OD pairs are sampled iteratively using population-weighted probabilities until the running mean travel time converges (change < 1% = 0.01 seconds).
3. All OD routes, lengths, and travel times are saved to CSV.

### Convergence Criterion
The loop continues sampling until:
```
|mean(t_1..n) − mean(t_1..n-1)| < 0.01 seconds
```
This ensures the estimated mean travel time has stabilised before moving to the next network.

### Key Parameters
| Parameter | Value | Description |
|-----------|-------|-------------|
| `epsilon` threshold | 0.01 s | Convergence tolerance for mean travel time |
| Graph directory | `L:/yiyi/graphs_cov_no_res3/` | Pre-built OSMnx graphs (no residential streets) |
| OD probability directory | `L:/yiyi/ODpts_propbability/` | Population-weighted grid point CSVs |
| Output directory | `L:/yiyi/dry_OD_routing/` | Per-network dry routing results |

In [ ]:
# --- Summary container ---
# Stores the converged mean dry travel time for each of the ~4190 sub-networks.
# Populated incrementally; rows with None remain for failed/skipped networks.
dry_mean_traveltime_df = pd.DataFrame(
    columns=['net_id','dry_mean_time_s'],
    index=range(4190)
    )
counter = 0 # Tracks successfully processed networks

for graph_file in tqdm(os.listdir('L:yiyi/graphs_cov_no_res3/')):

    # ── Load graph ──────────────────────────────────────────────────────────
    # Each graph is a pickled OSMnx/NetworkX directed graph representing one
    # sub-network (identified by net_id parsed from the filename).
    G1 =  pickle.load(open('L:yiyi/graphs_cov_no_res3/'+graph_file, 'rb'))
    net_id = graph_file[15:][:-3]
    
    try:
        # ── Add speed and travel_time on each graph edge ─────────────────────
        G1_speed = ox.speed.add_edge_speeds(G1)
        G1_speed_time = ox.speed.add_edge_travel_times(G1_speed)

        # ── Load OD sampling probability table ───────────────────────────────
        # Each row is a grid cell with:
        #   pointid   – unique cell identifier
        #   grid_code – population count used as sampling weight
        #   POINT_X   – longitude (mislabelled as lat in variable names below)
        #   POINT_Y   – latitude
        # Note: net_id is 1-indexed; probability file index is 0-indexed (net_id - 1).
        OD_probability_1 = pd.read_csv(r'L:\yiyi\ODpts_propbability\n_4229_bnd_' + str(int(net_id)-1) + '.txt')
        pt_id_lst = list(OD_probability_1['pointid'])
        pt_prob_lst = list(OD_probability_1['grid_code'])

        # ── Initialise per-network OD result table ─────────────────────────── 
        OD_df = pd.DataFrame(columns=['index',
                                      'O_node_id',           # OSM node ID of snapped origin
                                      'D_node_id',           # OSM node ID of snapped destination
                                      'O_lat',               # Origin longitude (POINT_X column, see note above)
                                      'O_lon',               # Origin latitude  (POINT_Y column)
                                      'D_lat',               # Destination longitude
                                      'D_lon',               # Destination latitude
                                      'OD_route',            # Ordered list of OSM node IDs forming the route
                                      'OD_route_length_m',   # Total route length in metres
                                      'OD_route_time_s'      # Total route travel time in seconds
                                      ])   
        index = 0
        epsilon = float('inf')  # Forces at least one iteration before convergence check
        travel_time_lst = []    # Running list of sampled travel times for convergence
       
        # ── Monte Carlo OD sampling loop ─────────────────────────────────────
        # Continues until the running-mean travel time converges.
        while epsilon > 0.01:

            #Sample a point based on probability
            sample_O_pt = random.choices(pt_id_lst, pt_prob_lst)[0]
            sample_D_pt = random.choices(pt_id_lst, pt_prob_lst)[0]
            while sample_O_pt == sample_D_pt:
                sample_D_pt = random.choices(pt_id_lst, pt_prob_lst)[0]

            # O
            sample_O_lat = OD_probability_1.loc[OD_probability_1['pointid'] == sample_O_pt, 'POINT_X'].iloc[0]
            sample_O_lon = OD_probability_1.loc[OD_probability_1['pointid'] == sample_O_pt, 'POINT_Y'].iloc[0]
            # D
            sample_D_lat = OD_probability_1.loc[OD_probability_1['pointid'] == sample_D_pt, 'POINT_X'].iloc[0]
            sample_D_lon = OD_probability_1.loc[OD_probability_1['pointid'] == sample_D_pt, 'POINT_Y'].iloc[0]

            O_node_id = ox.distance.get_nearest_node(G1, (sample_O_lon, sample_O_lat), method='haversine') # lon, lat
            D_node_id = ox.distance.get_nearest_node(G1, (sample_D_lon, sample_D_lat), method='haversine')

            # Compute shortest path (by travel time) between snapped nodes.
            # Skips this OD pair if no path exists (e.g. disconnected subgraph).
            try:
                OD_route = nx.shortest_path(G1_speed_time, O_node_id, D_node_id, weight='travel_time')
            except:
                continue # No path found; sample a new OD pair
            
            # Aggregate route-level metrics from individual edge attributes
            OD_route_length = sum(ox.utils_graph.get_route_edge_attributes(G1_speed_time, OD_route, 'length'))      # meters
            OD_route_time =sum(ox.utils_graph.get_route_edge_attributes(G1_speed_time, OD_route, 'travel_time'))    # seconds

            # Append results to the OD table
            OD_df.loc[index] = [index+1, O_node_id, D_node_id, sample_O_lat, sample_O_lon, sample_D_lat, sample_D_lon,
                            OD_route, OD_route_length, OD_route_time]

            travel_time_lst.append(OD_route_time)

            # ── Convergence check ────────────────────────────────────────────
            # Skip check on first iteration (need at least two values).
            # epsilon = absolute change in running mean after adding latest sample.
            if len(travel_time_lst) == 1:
                epsilon = float('inf')
            else:
                epsilon = abs(np.mean(travel_time_lst) - np.mean(travel_time_lst[:-1]))

            index += 1
            # Save incrementally so partial results are preserved if the process is interrupted
            OD_df.to_csv('L:/yiyi/dry_OD_routing/G_' + net_id + '_dry.csv')
        
        # ── Record converged mean for this network ───────────────────────────
        dry_mean_traveltime_df.at[counter, 'net_id'] = net_id
        dry_mean_traveltime_df.at[counter, 'dry_mean_time_s'] = np.mean(travel_time_lst)
    
        counter += 1
    except:
        # Print the network ID so failed networks can be re-run or investigated
        print(net_id)

## Section 3. Wet Condition Routing

This section re-routes the dry OD pairs on flood-disrupted networks across **10 return periods (RP)**:
`5, 10, 20, 50, 75, 100, 200, 250, 500, 1000` years.

For each network and each RP:
- The flood-disrupted graph (edges removed/penalised where flood depth exceeds 30 mm) is loaded.
- Each OD pair from the dry simulation is re-routed on the disrupted graph.
- If no path exists (complete isolation due to flooding), the row is left at zero/None.

The outputs allow direct comparison of dry vs. wet travel times for resilience metrics (e.g. travel-time ratio, accessibility loss).

### Flood Disruption Threshold
Edges with flood depth ≥ **30 mm** are removed from disrupted graphs (set in upstream preprocessing).

### Key Parameters
| Parameter | Value | Description |
|-----------|-------|-------------|
| Return periods | 5–1000 years | Ten flood frequency scenarios |
| Disrupted graph directory | `L:/yiyi/2576_disrupted_graphs_30mm/` | Flood-disrupted graphs by RP |
| Dry routing directory | `L:/yiyi/dry_OD_routing/` | Input: dry OD CSVs from Section 2 |
| Output directory | `L:/yiyi/2576_dry_wet_OD_routing/` | Combined dry + wet routing results |

### Output Schema (per row)
Each output CSV contains all dry columns plus, for each RP `{rp}`:
- `OD_route_FUP_{rp}` — re-routed node sequence under flooding
- `OD_length_FUP_{rp}` — re-routed path length (m); 0 if no path
- `OD_time_FUP_{rp}` — re-routed travel time (s); 0 if no path

In [ ]:
# ── Step 1: Identify networks with complete flood-disrupted graph coverage ──
# Only networks present across ALL 10 return periods are included to ensure
# a consistent comparison set in downstream analysis.

RP_lst = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000] # Return periods (years)

flood_graphs_lst = []

for rp in RP_lst:
    one_lst = []
    for file in os.listdir('L:/yiyi/2576_disrupted_graphs_30mm/FUP_' + str(rp) + '/'):
        
        # Filename format: 'G_{net_id}_disrupted_FUP_{rp}.pk'
        # Slice off 'G_' prefix and the trailing '_disrupted_FUP_{rp}.pk'
        net_id = int(file[2:][:(-18) - len(str(rp))])
        one_lst.append(net_id)
    flood_graphs_lst.append(one_lst)

# Intersection: retain only networks that have disrupted graphs for every RP
common_graphs = set(flood_graphs_lst[0])
for lst in flood_graphs_lst[1:]:
    common_graphs.intersection_update(lst)

print('Number of common graphs:',len(common_graphs))

RP_lst = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000]

# ── Step 2: Re-route each dry OD pair on flood-disrupted graphs ─────────────
dry_routing_dir = 'L:/yiyi/dry_OD_routing/'

# for file in os.listdir(dry_routing_dir):
for net in tqdm(common_graphs):

    # Load dry OD pairs for this network (output of Section 2)
    dry_OD_df = pd.read_csv(dry_routing_dir + 'G_' + str(net) + '_dry.csv', index_col=0)
    net_id = net

    # Initialise wet routing table as a copy of dry results.
    # New columns for each RP are added below; defaults of 0 / None indicate
    # OD pairs where routing failed (no path on the flooded network).    
    wet_OD_df = dry_OD_df.copy()
    # Initiate flooded columns
    for rp in RP_lst:
        wet_OD_df['OD_route_FUP_' + str(rp)] = None     # Node sequence or None   
        wet_OD_df['OD_length_FUP_' + str(rp)] = 0       # Metres; 0 = no path / not computed
        wet_OD_df['OD_time_FUP_' + str(rp)] = 0         # Seconds; 0 = no path / not computed
    
    # ── Per-RP routing ────────────────────────────────────────────────────
    for rp in RP_lst:

        # Load the flood-disrupted graph for this network and return period.
        # Disrupted graphs have flood-inundated edges removed (depth ≥ 30 mm threshold).
        graph_disrupted = pickle.load(open('L:/yiyi/2576_disrupted_graphs_30mm/FUP_'+ str(rp) + '/G_' + 
                                               str(net_id) + '_disrupted_FUP_' + str(rp) + '.pk', 'rb'))
        print(rp)   # Progress indicator for inner loop

        # Re-route each dry OD pair on the flooded network
        for row_idx in range(wet_OD_df.shape[0]):
            O_node_id = wet_OD_df.iloc[row_idx]['O_node_id']
            D_node_id = wet_OD_df.iloc[row_idx]['D_node_id']

            try:
                # Shortest path on disrupted graph (travel_time weight).
                # Raises NetworkXNoPath if O and D are in disconnected components.
                wet_OD_route = nx.shortest_path(graph_disrupted, O_node_id, D_node_id, weight='travel_time')
                wet_OD_route_length = sum(ox.utils_graph.get_route_edge_attributes(graph_disrupted, wet_OD_route, 'length')) # meters
                wet_OD_route_time =sum(ox.utils_graph.get_route_edge_attributes(graph_disrupted, wet_OD_route, 'travel_time')) # seconds

                wet_OD_df.at[row_idx, 'OD_route_FUP_' + str(rp)] = wet_OD_route
                wet_OD_df.at[row_idx, 'OD_length_FUP_' + str(rp)] = wet_OD_route_length
                wet_OD_df.at[row_idx, 'OD_time_FUP_' + str(rp)] = wet_OD_route_time
                
            except:
                # No path exists between this OD pair on the flooded network.
                # Values remain at 0 / None; this constitutes full inaccessibility
                # and should be handled explicitly in downstream resilience metrics.
                continue
    
    # Save combined dry + wet routing results for this network
    wet_OD_df.to_csv('L:yiyi/2576_dry_wet_OD_routing/G_' + str(net) + '_dry_wet_routing.csv')